# Multi-PDF -> Markdown con marker-pdf (chunking automatico)

Convierte un lote de PDFs a Markdown usando [marker-pdf](https://github.com/datalab-to/marker), con OCR forzado, salida `.zip` por documento, y **chunking automatico** para PDFs grandes que no entran en GPU en una sola pasada.

**Carpetas en Drive:**
- Entrada: `MyDrive/PDF2MD/` (los PDFs)
- Salida: `MyDrive/PDF2MD/Output/` (un `.zip` por PDF con el `.md`, las imagenes y el `_meta.json`)

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Como funciona

1. Cuenta las paginas de cada PDF (`pypdf`).
2. **PDF chico (<=100 paginas):** lo procesa en una sola pasada con `PdfConverter`, igual que siempre.
3. **PDF grande (>100 paginas):** lo divide automaticamente en tandas de 100 paginas (`0-99`, `100-199`, ...; **la ultima va hasta el final sola, no la ajustas a mano**), procesa cada tanda reutilizando los modelos ya cargados, y une los `.md` en orden. A las imagenes les antepone `p{tanda}_` para que no colisionen los nombres y reescribe las referencias en el markdown.

**Reanudable:** los `.zip` ya generados en Drive se saltean, asi que si se corta el runtime relanzas y sigue donde quedo.


## 1) Setup + Drive + listado

Corre esta celda. Instala `marker-pdf` (largo la primera vez por sus modelos), monta Drive, lista los PDFs pendientes.


In [ ]:
get_ipython().system("pip install marker-pdf pypdf -q")

from google.colab import drive
drive.mount("/content/drive")

# === RUTAS (editar si tus carpetas se llaman distinto) ===
SOURCE_DIR = "/content/drive/MyDrive/PDF2MD"
DEST_DIR   = "/content/drive/MyDrive/PDF2MD/Output"

# === CONFIG ===
FORCE_OCR        = True
PAGES_PER_CHUNK  = 100   # PDFs con mas paginas que esto se trocean automaticamente
BATCH_SIZES = {
    "recognition_batch_size": 96,
    "detection_batch_size":   24,
    "layout_batch_size":      24,
    "equation_batch_size":    24,
    "table_rec_batch_size":   24,
}

import os
os.makedirs(DEST_DIR, exist_ok=True)
pdfs = sorted(f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(".pdf"))
done = sorted(f for f in os.listdir(DEST_DIR) if f.lower().endswith(".zip"))
print(f"PDFs en origen:   {len(pdfs)}")
print(f"Ya convertidos:   {len(done)}")
print(f"Pendientes:       {len(pdfs) - len(done)}")


## 2) Procesar (carga modelos una sola vez, reusa entre PDFs)

Corre esta celda. Procesa todos los PDFs pendientes en `PDF2MD/`, troceando los que superen `PAGES_PER_CHUNK`. Suena un ruidito al terminar.


In [ ]:
import os, re, shutil, json, time
import numpy as np
from IPython.display import Audio, display
from pypdf import PdfReader

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered
from marker.settings import settings

RED, GREEN, YELL, RESET = "\033[91m", "\033[92m", "\033[93m", "\033[0m"
WORK = "/content/work"

# === Cargar modelos UNA sola vez (persisten entre re-runs de la celda) ===
if "MODELS" not in globals():
    print("Cargando modelos de marker (una sola vez)...")
    settings.OUTPUT_IMAGE_FORMAT = "png"   # conserva imagenes, evita el bug del JPEG
    MODELS = create_model_dict()
    print("Modelos listos.\n")

def make_converter(page_range=None):
    """Construye un PdfConverter con la config + un page_range opcional.
    REUSA los modelos cargados (MODELS) -> es rapido, no recarga nada."""
    cfg = {
        "output_format": "markdown",
        "force_ocr":     FORCE_OCR,
        **BATCH_SIZES,
    }
    if page_range is not None:
        cfg["page_range"] = page_range
    cp = ConfigParser(cfg)
    return PdfConverter(
        config=cp.generate_config_dict(),
        artifact_dict=MODELS,
        processor_list=cp.get_processors(),
        renderer=cp.get_renderer(),
        llm_service=cp.get_llm_service(),
    )

def plan_chunks(n_pages, chunk_size):
    """Devuelve [(start, end), ...] 0-indexado e inclusivo. La ultima tanda
    llega hasta n_pages-1 sola, sin ajuste manual."""
    ranges = []
    for start in range(0, n_pages, chunk_size):
        end = min(start + chunk_size - 1, n_pages - 1)
        ranges.append((start, end))
    return ranges

_IMG_REF = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")
def _prefix_images(text, images, prefix):
    """Renombra las imagenes con un prefijo de tanda para que no colisionen
    entre chunks, y reescribe las referencias ![](nombre) del markdown.
    Devuelve (text_reescrito, images_renombradas)."""
    rename = {name: f"{prefix}{name}" for name in (images or {})}
    if not rename:
        return text, {}
    def _sub(m):
        alt, src = m.group(1), m.group(2)
        return f"![{alt}]({rename.get(src, src)})"
    new_text = _IMG_REF.sub(_sub, text)
    new_images = {rename[k]: v for k, v in images.items()}
    return new_text, new_images

def process_pdf(stem, pdf_path):
    """Procesa un PDF y devuelve (text, images_dict, metadata) listos para
    serializar. Trocea automaticamente si supera PAGES_PER_CHUNK."""
    reader = PdfReader(pdf_path)
    n_pages = len(reader.pages)

    if n_pages <= PAGES_PER_CHUNK:
        print(f"  paginas: {n_pages} -> una sola pasada")
        rendered = make_converter()(pdf_path)
        text, _, images = text_from_rendered(rendered)
        metadata = getattr(rendered, "metadata", None)
        return text, dict(images or {}), metadata

    ranges = plan_chunks(n_pages, PAGES_PER_CHUNK)
    print(f"  paginas: {n_pages} -> {len(ranges)} tanda(s): "
          + ", ".join(f"{a}-{b}" for a, b in ranges))

    parts_text, parts_images, last_meta = [], {}, None
    for idx, (start, end) in enumerate(ranges, 1):
        t0 = time.time()
        page_range = f"{start}-{end}"
        print(f"   tanda {idx}/{len(ranges)} ({page_range})...", end=" ", flush=True)
        rendered = make_converter(page_range=page_range)(pdf_path)
        text, _, images = text_from_rendered(rendered)
        if not text or not text.strip():
            raise RuntimeError(f"tanda {idx} ({page_range}) no produjo texto")
        text_p, images_p = _prefix_images(text, images, prefix=f"p{idx}_")
        parts_text.append(text_p)
        parts_images.update(images_p)
        last_meta = getattr(rendered, "metadata", None) or last_meta
        print(f"OK ({time.time()-t0:.1f}s, {len(images_p)} img)")

    return "\n\n".join(parts_text), parts_images, last_meta

pdfs = sorted(f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(".pdf"))
print(f"{len(pdfs)} PDF(s) en origen\n")

ok, fallaron, saltados = [], [], []
for i, original in enumerate(pdfs, 1):
    stem = os.path.splitext(original)[0]
    final_zip = os.path.join(DEST_DIR, f"{stem}.zip")
    print(f"===== [{i}/{len(pdfs)}] {original} =====")

    if os.path.exists(final_zip):
        print(f"{YELL}  YA EXISTE en Drive, salteando{RESET}")
        saltados.append(original); continue

    item_dir = os.path.join(WORK, stem)
    shutil.rmtree(item_dir, ignore_errors=True)
    os.makedirs(item_dir)
    try:
        text, images, metadata = process_pdf(stem, os.path.join(SOURCE_DIR, original))

        if not text or not text.strip():
            raise RuntimeError("marker no produjo texto")

        # .md
        with open(os.path.join(item_dir, f"{stem}.md"), "w", encoding="utf-8") as fh:
            fh.write(text)

        # imagenes (mismos nombres que referencia el .md)
        for img_name, img in images.items():
            img.save(os.path.join(item_dir, img_name))

        # metadata
        if metadata is not None:
            with open(os.path.join(item_dir, f"{stem}_meta.json"), "w", encoding="utf-8") as fh:
                json.dump(metadata, fh, ensure_ascii=False, indent=2, default=str)

        # zip local -> copia atomica a Drive
        tmp_zip = shutil.make_archive(os.path.join(WORK, stem), "zip", item_dir)
        part = final_zip + ".part"
        shutil.copy(tmp_zip, part)
        os.replace(part, final_zip)
        os.remove(tmp_zip)

        print(f"{GREEN}  OK -> guardado en Drive: {stem}.zip{RESET}")
        ok.append(original)
    except Exception as e:
        print(f"{RED}  ERROR: {original} -> {e}{RESET}")
        fallaron.append(original)
    finally:
        shutil.rmtree(item_dir, ignore_errors=True)

print(f"\n========== RESUMEN ==========")
print(f"{GREEN}OK:        {len(ok)}{RESET}")
print(f"{YELL}Saltados:  {len(saltados)}{RESET}")
print(f"{RED}Fallaron:  {len(fallaron)}{RESET}")
for f in fallaron:
    print(f"{RED}  - {f}{RESET}")

# Ruidito final
sr = 22050
out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out_audio = np.concatenate([out_audio, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
